In [1]:
import os
import pandas as pd
import numpy as np

# Load dataset
data_path = os.path.join('data', 'all_months.csv')
df = pd.read_csv(data_path)

print(f"Dataset loaded successfully. Shape: {df.shape}")

Dataset loaded successfully. Shape: (798, 56)


In [2]:
# Clean column names (strip whitespace)
df.columns = df.columns.str.strip()

print("Headers cleaned.")

Headers cleaned.


In [3]:
# Drop 'Naew' columns if present
cols_to_drop = ['Naew']

# Drop safely without raising an error if the column doesn't exist
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print(f" Column 'mnaew' removed. Updated DataFrame shape: {df_clean.shape}")

 Column 'mnaew' removed. Updated DataFrame shape: (798, 55)


In [4]:
# Map months in true chronological order (Shrawan 2082 = 1 to Baishakh 2083 = 10)
bs_month_map = {
    'Shrawan': 1, 'Bhadra': 2, 'Ashwin': 3, 'Kartik': 4, 
    'Mangsir': 5, 'Poush': 6, 'Magh': 7, 'Falgun': 8, 
    'Chaitra': 9, 'Baishakh': 10
}

if 'month_name' in df.columns:
    df['month_idx'] = df['month_name'].map(bs_month_map)

# Ensure data is sorted chronologically
df = df.sort_values(['Product_Name', 'month_idx']).reset_index(drop=True)
print("BS Calendar mapped and sorted chronologically.")

BS Calendar mapped and sorted chronologically.


In [5]:
# Strip whitespace from Product_Name
df['Product_Name'] = df['Product_Name'].str.strip()

# Apply custom product name mapping
product_mapping = {
    "Brocauli": "Broccoli",
    "Dragonfruits": "Dragon_Fruits",
    "Gunduruk": "Gundruk",
    "Orange (Sweet)": "Orange_Sweet",
    "Sponge_Groud": "Sponge_Gourd",
    "Sajiwan_Swigan": "Sajiwan",
    "Tomato-Big": "Tomato_Big",
}

df['Product_Name'] = df['Product_Name'].replace(product_mapping)

# Standardize Category
if 'Category' in df.columns:
    df['Category'] = df['Category'].str.strip().str.lower()
    df['Category'] = df['Category'].replace({'fruits': 'fruit', 'vegetables': 'vegetable'})

# Sort chronologically by product and month_idx
df = df.sort_values(['Product_Name', 'month_idx']).reset_index(drop=True)

print("Product mapping applied and data sorted.")

Product mapping applied and data sorted.


In [6]:
# Ensure Unit is a clean string
df['Unit'] = df['Unit'].str.strip().str.lower()

# Find most frequent unit per product
unit_mode = df.groupby('Product_Name')['Unit'].agg(lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0])
df['unit_canonical'] = df['Product_Name'].map(unit_mode)

# Flag products where unit changed across months
df['unit_changed'] = df['Unit'] != df['unit_canonical']

print(f"Unit canonicalized. Products with unit changes: {df['unit_changed'].sum()}")

Unit canonicalized. Products with unit changes: 0


In [7]:
# Price columns
price_cols = ['Min_Price', 'Max_Price', 'Avg_Price']

# Replace 0 values with NaN across price columns
for col in price_cols:
    if col in df.columns:
        # Convert 0 or <= 0 to NaN
        df[col] = df[col].replace(0, np.nan)
        df.loc[df[col] <= 0, col] = np.nan

print("Price processing complete: 0 prices converted to NaN without imputation.")
print(df[price_cols].isnull().sum())

Price processing complete: 0 prices converted to NaN without imputation.
Min_Price    77
Max_Price    76
Avg_Price     0
dtype: int64


In [8]:
# Identify district/origin columns dynamically
metadata_cols = [
    'Product_Name', 'Category', 'bs_year', 'bs_month', 'month_idx', 'month_name',
    'Volume', 'Min_Price', 'Max_Price', 'Avg_Price', 'Unit', 'unit_canonical',
    'unit_changed', 'Total_Amount', 'Volume_Equals', 'TOTAL_sources',
    'reconciliation_gap', 'import_share', 'n_months_present', 'is_balanced'
]

district_cols = [c for c in df.columns if c not in metadata_cols]

# Recompute sum of all district/source columns
df['TOTAL_sources'] = df[district_cols].fillna(0).sum(axis=1)

# Calculate reconciliation gap (Volume - TOTAL_sources)
df['reconciliation_gap'] = df['Volume'] - df['TOTAL_sources']

# Flag whether Volume matches TOTAL_sources exactly
df['Volume_Equals'] = df['Volume'] == df['TOTAL_sources']

print(f"TOTAL_sources recomputed.")
print(f"Rows where Volume matches TOTAL_sources: {df['Volume_Equals'].sum()} / {len(df)}")

TOTAL_sources recomputed.
Rows where Volume matches TOTAL_sources: 775 / 798


In [9]:
# Compute import share: (India + China + Bhutan) / TOTAL_sources
import_cols = [col for col in ['India', 'China', 'Bhutan'] if col in df.columns]

if import_cols:
    import_sum = df[import_cols].fillna(0).sum(axis=1)
    df['import_share'] = np.where(df['TOTAL_sources'] > 0, import_sum / df['TOTAL_sources'], np.nan)

print("Import share calculated.")

Import share calculated.


In [10]:
# Count how many distinct months each product appears in
product_counts = df.groupby('Product_Name')['month_idx'].nunique()
df['n_months_present'] = df['Product_Name'].map(product_counts)

# Flag balanced products (present in all 10 months)
df['is_balanced'] = df['n_months_present'] == 10

print(f"Total Unique Canonical Products: {df['Product_Name'].nunique()}")
print(f"Balanced Panel Products (present in all 10 months): {df[df['is_balanced']]['Product_Name'].nunique()}")

Total Unique Canonical Products: 95
Balanced Panel Products (present in all 10 months): 0


In [11]:
# Export cleaned dataset
output_path = os.path.join('data', 'all_months_clean.csv')
df.to_csv(output_path, index=False)

print(f" Cleaned dataset exported to '{output_path}'")
print(f"Final Shape: {df.shape[0]} rows, {df.shape[1]} columns")

 Cleaned dataset exported to 'data/all_months_clean.csv'
Final Shape: 798 rows, 56 columns
